<a href="https://colab.research.google.com/github/AdityaR28/OSS4008A---Introduction-to-Remote-Sensing-and-Applications/blob/main/students/devon_clermont-gauer/code/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Water Dashboard (Project 11)
#**Name:** Devon Clermont-Gauer

#Interactive map of water bodies in Canada. Reference layers (all, permanent and seasonal water from JRC; rivers from HydroSHEDS) can be toggled with the buttons, and **AI water detection** runs the trained U-Net on Sentinel-2 imagery inside a box the user draws. The chat panel is a placeholder for a future AI assistant.

#**How to run:**
#1. Replace `YOUR-PROJECT-ID` in Cell 1 with your Google Earth Engine Cloud project ID.
#2. Make sure the trained model and band statistics are in your Google Drive (paths in Cell 4).
#3. Runtime → Run all (a T4 GPU runtime makes AI detection faster, but CPU works).

In [2]:
# Cell 1: install libraries and log in to Earth Engine
!pip install -q geemap segmentation-models-pytorch

import ee
import geemap

ee.Authenticate()
ee.Initialize(project='causal-sky-509317-u9')                    # replace with your Cloud project ID
print('Earth Engine says:', ee.String('hello').getInfo())   # connection test

Earth Engine says: hello


In [3]:
# Cell 2: download this project's code from the course repo (public, no login needed)
import os, sys, subprocess

REPO     = 'OSS4008A---Introduction-to-Remote-Sensing-and-Applications'
REPO_DIR = f'/content/{REPO}'
CODE_DIR = f'{REPO_DIR}/students/devon_clermont-gauer/code'

if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)           # update to latest
else:
    subprocess.run(['git', 'clone', '--depth', '1',
                    f'https://github.com/AdityaR28/{REPO}.git', REPO_DIR], check=True)

sys.path.insert(0, CODE_DIR)                                 # let Python find the modules
for f in ['water_layers.py', 'water_ai.py', 'water_classify.py', 'band_stats.json']:
    print(f, 'found' if os.path.exists(f'{CODE_DIR}/{f}') else 'MISSING')

water_layers.py found
water_ai.py found
water_classify.py found
band_stats.json found


In [4]:
# Cell 3: download the trained model (shared on Google Drive; too large for the repo)
import gdown

MODEL_ID   = '1ZvIHmtM5tdOvZHpJC5JdmKNEVgkQbkL4'    # from the model's Drive share link
MODEL_PATH = '/content/unet_resnet34_best.pth'
STATS_PATH = f'{CODE_DIR}/band_stats.json'          # band statistics, stored in the repo

if not os.path.exists(MODEL_PATH):
    gdown.download(id=MODEL_ID, output=MODEL_PATH, quiet=False)
print('Model ready:', os.path.exists(MODEL_PATH))

Downloading...
From (original): https://drive.google.com/uc?id=1ZvIHmtM5tdOvZHpJC5JdmKNEVgkQbkL4
From (redirected): https://drive.google.com/uc?id=1ZvIHmtM5tdOvZHpJC5JdmKNEVgkQbkL4&confirm=t&uuid=4e7a25e4-6ec8-40fb-bc58-d1ca904bfd2c
To: /content/unet_resnet34_best.pth
100%|██████████| 97.9M/97.9M [00:01<00:00, 76.6MB/s]

Model ready: True


In [9]:
# Cell 4: dashboard UI (chat panel | map, grouped toolbar underneath)
import ee, math, base64, io
import numpy as np
import ipywidgets as widgets
from PIL import Image
from ipyleaflet import DrawControl, ImageOverlay, TileLayer, WidgetControl
from water_layers import get_overview_layer, get_rivers_layer, canada
from water_ai import load_model, detect_water_ai
from water_classify import classify_water, class_rgba, summary_text, LAKE, RIVER, POND

# Load the trained U-Net (downloaded in Cell 3) and the band statistics (from the repo)
load_model(MODEL_PATH, STATS_PATH)

LIMIT_KM2 = 20000         # largest area the AI will scan (matches max_km2 in water_ai.detect_water_ai)
ALL_TYPES = (LAKE, RIVER, POND)

# --- Styling (Google-like: Roboto, cards, pill buttons, chat bubbles, legend) ---
css = widgets.HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Roboto:wght@400;500&display=swap');
.app       {font-family: Roboto, Arial, sans-serif; background:#f8f9fa;
            border-radius:16px; padding:16px;}
.title     {font-size:22px; font-weight:500; color:#202124; margin:0 0 12px 4px;}
.card      {background:#fff; border-radius:12px; padding:12px;
            box-shadow:0 1px 3px rgba(60,64,67,.3), 0 4px 8px rgba(60,64,67,.15);}
.map-card  {border-radius:12px; overflow:hidden; padding:0;
            box-shadow:0 1px 3px rgba(60,64,67,.3), 0 4px 8px rgba(60,64,67,.15);}
.chat-log  {height:480px; overflow-y:auto; display:flex; flex-direction:column-reverse;}
.chat-log .widget-html-content {display:flex; flex-direction:column; margin-bottom:auto;}
.bot  {background:#e8f0fe; color:#202124; border-radius:16px 16px 16px 4px;
       padding:8px 12px; margin:4px 0; max-width:85%; width:fit-content;}
.user {background:#1a73e8; color:#fff; border-radius:16px 16px 4px 16px;
       padding:8px 12px; margin:4px 0 4px auto; max-width:85%; width:fit-content;}
.pill.widget-button, .pill.widget-toggle-button {border-radius:20px !important; background:#fff !important;
       color:#1a73e8 !important; border:1px solid #dadce0 !important;
       font-weight:500; height:36px; padding:0 14px; margin-right:6px;}
.pill.widget-button:hover {background:#f1f3f4 !important;}
.pill.active.widget-button, .pill.mod-active {background:#1a73e8 !important; color:#fff !important;
       border-color:#1a73e8 !important;}
.chat-input input {border-radius:20px !important; border:1px solid #dadce0 !important;
       padding:0 14px !important; height:36px;}
.leaflet-control-attribution {font-size:9px !important; opacity:0.5;
       background:transparent !important; line-height:1.2;}
.leaflet-control-attribution:hover {opacity:1;}
.ai-legend {background:#fff !important; color:#202124 !important; opacity:1 !important;
            padding:8px 12px; border-radius:8px; font:13px Roboto, Arial, sans-serif;
            box-shadow:0 1px 4px rgba(0,0,0,.35);}
.ai-legend * {color:#202124 !important; opacity:1 !important;}
.ai-legend-title {font-weight:500; margin-bottom:4px;}
.ai-legend-row {display:flex; align-items:center; margin-top:4px;}
.ai-legend-swatch {width:14px; height:14px; border-radius:3px; margin-right:8px;
                   border:1px solid rgba(0,0,0,.25);}
</style>""")

def bubble(text, who):
    """Wrap a message in a chat bubble ('bot' = assistant, 'user' = user)."""
    return f'<div class="{who}">{text}</div>'

def say(text):
    """Post an assistant message in the chat."""
    chat_log.value += bubble(text, 'bot')

# --- Shared state (defined before the map, which uses it) ---
drawn = {'box': None, 'ctrl': None}                    # last drawn rectangle and the draw tool
scan = {'busy': False, 'last': None, 'gen': 0,          # scanning state for Scan view / Auto AI
        'auto': False, 'warned': False}
result = {'water': None, 'classes': None, 'bounds': None, 'res': None, 'legend': None}  # last AI result

# --- Helpers for view scanning ---
def view_box(m):
    """Current view as (south, west, north, east), or None if the map isn't ready."""
    if not m.bounds:
        return None
    (s, w), (n, e) = m.bounds
    return s, w, n, e

def box_km2(b):
    """Approximate ground area of a (south, west, north, east) box in km²."""
    s, w, n, e = b
    return (n - s) * 111 * (e - w) * 111 * math.cos(math.radians((s + n) / 2))

def overlap(a, b):
    """Fraction of box b that lies inside box a."""
    s, w = max(a[0], b[0]), max(a[1], b[1])
    n, e = min(a[2], b[2]), min(a[3], b[3])
    if n <= s or e <= w:
        return 0.0
    return box_km2((s, w, n, e)) / box_km2(b)

def expected_res(b):
    """Resolution (m) the AI would use for this view: same rule as water_ai.detect_water_ai
    (10 m up to 400 km², coarser above, up to 60 m)."""
    return min(60, max(10, 10 * math.sqrt(box_km2(b) / 400)))

def on_map_move(change):
    """Auto AI: the map reports its new bounds when a pan or zoom finishes, so scan then.
    (Timers don't run in Colab, so there is no extra delay; the guards in water_in_view
    skip scans that are too large, barely moved, or overlapping a running scan.)"""
    if scan['auto']:
        water_in_view(None, auto=True)

# --- Drawing AI results on the map ---
def rgba_overlay(rgba, bounds, name='Water (AI)'):
    """Turn an RGBA picture into a map layer placed at bounds."""
    buf = io.BytesIO()
    Image.fromarray(rgba).save(buf, format='PNG')
    url = 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()
    return ImageOverlay(url=url, bounds=bounds, name=name)

# --- Place labels: provinces, cities, parks, etc. (Esri reference layer, text only) ---
LABELS_URL = ('https://server.arcgisonline.com/ArcGIS/rest/services/Reference/'
              'World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}')

def labels_layer():
    """Transparent label layer; labels appear and disappear by zoom level (no overcrowding)."""
    return TileLayer(url=LABELS_URL, name='Place labels', attribution='Esri', max_zoom=19)

def labels_on_top(m):
    """Move place labels above everything else so AI results never hide the names.
    A fresh layer is added: re-adding a removed layer fails in ipyleaflet."""
    for layer in list(m.layers):
        if layer.name == 'Place labels':
            m.remove_layer(layer)
    m.add(labels_layer())

# --- Legend for AI results (bottom-right of the map, updates with each scan/filter) ---
LEGEND_ITEMS = {LAKE: ('Lakes', '#1e6eff'), RIVER: ('Rivers', '#00e5ff'), POND: ('Ponds', '#ff3ec8')}

def update_legend(m, types=None, classified=True):
    """Replace the AI legend: one row per shown type, or 'All water' for unclassified scans."""
    old = result.get('legend')
    if old is not None and any(c is old for c in m.controls):   # only remove our own legend, if present
        m.remove_control(old)
    result['legend'] = None
    if result['water'] is None:
        return                                          # nothing shown: no legend
    if classified:
        rows = [LEGEND_ITEMS[k] for k in types]
        title = 'AI water types'
    else:
        rows = [('All water (rough)', '#00ffff')]
        title = 'AI water'
    html = (f'<div class="ai-legend">'
            f'<div class="ai-legend-title">{title}</div>' +
            ''.join(f'<div class="ai-legend-row"><span class="ai-legend-swatch" '
                    f'style="background:{c}"></span>{name}</div>'
                    for name, c in rows) + '</div>')
    result['legend'] = WidgetControl(widget=widgets.HTML(html), position='bottomright')
    m.add(result['legend'])

def show_result(types=ALL_TYPES):
    """Draw the last AI result. Classified results show only `types`, each in its own colour;
    unclassified (coarse) results show all water in cyan."""
    m = map_area.children[0]
    for layer in list(m.layers):                        # remove the previous AI result, if any
        if layer.name == 'Water (AI)':
            m.remove_layer(layer)
    if result['water'] is None:
        update_legend(m)                                # removes the legend
        return
    if result['classes'] is not None:
        rgba = class_rgba(result['classes'], show=types)
    else:
        rgba = np.zeros((*result['water'].shape, 4), np.uint8)
        rgba[result['water']] = (0, 255, 255, 200)
    m.add(rgba_overlay(rgba, result['bounds']))
    labels_on_top(m)                                    # keep place names readable over the result
    update_legend(m, types, classified=result['classes'] is not None)

# --- Map ---
def build_map(layer_fn=None):
    """Create a fresh map of Canada, optionally with one reference layer."""
    new_map = geemap.Map(center=[60, -95], zoom=4, height='560px',
                         basemap='Esri.WorldImagery',   # Esri (OpenStreetMap tiles are blocked in Colab)
                         draw_ctrl=False)               # use our own rectangle tool instead of geemap's
    new_map.addLayer(canada.style(color='ffffff', fillColor='00000000', width=1),
                     {}, 'Canada boundary')
    if layer_fn:
        img, vis, name = layer_fn()
        new_map.addLayer(img, vis, name)
        new_map.add_legend(title='Layer', legend_dict={name: '#' + vis.get('palette', ['00e5ff'])[0]})
    new_map.add(labels_layer())                         # province, city and place names (on top)

    # Rectangle drawing tool: the last drawn box is stored for AI detection
    draw = DrawControl(rectangle={'shapeOptions': {'color': '#ffd400', 'weight': 2}},
                       polygon={}, polyline={}, circlemarker={}, marker={}, circle={})
    def on_draw(target, action, geo_json):
        if action == 'created':
            drawn['box'] = ee.Geometry(geo_json['geometry'])
        elif action == 'deleted':
            drawn['box'] = None
    draw.on_draw(on_draw)
    new_map.add(draw)
    drawn['box'], drawn['ctrl'] = None, draw

    new_map.observe(on_map_move, names='bounds')        # for Auto AI
    new_map.layout.width = '100%'
    return new_map

# Container so the map can be swapped out when a button reloads it
map_area = widgets.Box([build_map()], layout=widgets.Layout(width='70%', margin='0 0 0 12px'))
map_area.add_class('map-card')

# --- Chat panel (placeholder until an AI assistant is connected) ---
chat_log = widgets.HTML(
    bubble('Hi! Zoom in and click "Scan view" to detect water with the trained model, or draw a box '
           'with the square tool. Close-up scans (up to ~1,600 km²) also sort the water into '
           '<b style="color:#1e6eff">lakes</b>, <b style="color:#00b8d4">rivers</b> and '
           '<b style="color:#ff3ec8">ponds</b>; then use the Filter buttons to show one type.', 'bot')
    + bubble('Base map has loaded.', 'bot'))
chat_log.add_class('chat-log')

chat_in = widgets.Text(placeholder='Ask about water in Canada...',
                       layout=widgets.Layout(width='75%'))
chat_in.add_class('chat-input')
send_btn = widgets.Button(description='Send')
send_btn.add_class('pill')

def handle_message(_):
    """Show the user's message and a placeholder reply."""
    if chat_in.value.strip():
        chat_log.value += bubble(chat_in.value, 'user')
        say('The chat assistant is not connected yet.')
        chat_in.value = ''

send_btn.on_click(handle_message)
chat_panel = widgets.VBox([widgets.HTML('<div class="title">Assistant</div>'),
                           chat_log, widgets.HBox([chat_in, send_btn])],
                          layout=widgets.Layout(width='30%'))
chat_panel.add_class('card')

# --- Reference layer buttons: (label, icon, what to load) ---
layer_buttons = [
    ('Base map',              'globe', None),
    ('All water (reference)', 'tint',  get_overview_layer),
    ('Rivers (reference)',    'road',  get_rivers_layer),
]
btns = []

def make_handler(btn, label, layer_fn):
    """Build a click handler that reloads the map with this button's layer."""
    def handler(_):
        for b in btns:
            b.remove_class('active')
        btn.add_class('active')
        say(f'Loading {label.lower()}...')
        try:
            map_area.children = (build_map(layer_fn),)
            scan['last'] = None                         # new map: nothing scanned yet
            result['water'] = None                      # old AI result belongs to the old map
            say(f'{label} has loaded.')
        except Exception as e:
            say(f'Could not load {label.lower()}: {e}')
    return handler

for label, icon, fn in layer_buttons:
    b = widgets.Button(description=label, icon=icon)
    b.add_class('pill')
    b.on_click(make_handler(b, label, fn))
    btns.append(b)
btns[0].add_class('active')

# --- AI detection + classification, shared by the box, Scan view and Auto AI ---
def run_ai(region, where, gen=None):
    """Run the U-Net on a region, classify the water (if detailed enough) and show it."""
    say(f'Running AI water detection on the {where}...')
    try:
        water, bounds, km2, res = detect_water_ai(region)
        if gen is not None and gen != scan['gen']:
            return                                      # user moved on during the scan: discard
        classified = classify_water(water, res)         # None if too coarse to classify
        result.update(water=water, bounds=bounds, res=res,
                      classes=classified[0] if classified else None)
        set_filter(None)                                # new result: show all types
        msg = f'AI found {km2:.2f} km² of water in the {where} at {res} m resolution.'
        if classified:
            msg += f'<br>{summary_text(classified[1])}.'
        else:
            msg += ' Shown as all water (rough): zoom in closer (≤ ~1,600 km²) to sort it into lakes, rivers and ponds.'
        say(msg)
    except ValueError:                                  # area too large
        say(f'The {where} is too large for AI detection: zoom in or draw a smaller box.')
    except Exception as e:
        import traceback
        traceback.print_exc()                           # full error details below the cell
        say(f'Could not analyse the {where}: {e}')

def water_in_box(_):
    """Box button: scan the last drawn rectangle."""
    if drawn['box'] is None:
        say('Draw a rectangle on the map first (square tool on the left).')
        return
    if scan['busy']:
        say('A scan is already running; please wait.')
        return
    scan['busy'] = True
    scan['gen'] += 1
    try:
        run_ai(drawn['box'], 'box', gen=scan['gen'])
    finally:
        scan['busy'] = False

def water_in_view(_, auto=False):
    """Scan the area on screen. Auto scans skip quietly when not worthwhile."""
    m = map_area.children[0]
    b = view_box(m)
    if b is None:
        return
    if scan['busy']:
        if not auto:
            say('A scan is already running; please wait.')
        return
    if box_km2(b) > LIMIT_KM2:
        if not auto or not scan['warned']:
            say(f'Zoom in for AI detection (view is about {box_km2(b):,.0f} km²; the limit is {LIMIT_KM2:,} km²).')
            scan['warned'] = True
        return
    scan['warned'] = False
    if auto and scan['last'] and overlap(scan['last'], b) > 0.9:
        finer = result['res'] is not None and expected_res(b) < result['res'] - 1
        if not finer:
            return                                      # barely moved and no more detail to gain: skip
        # zoomed in on an already-scanned area: rescan for finer detail
    scan['busy'], scan['last'] = True, b
    scan['gen'] += 1
    try:
        run_ai(ee.Geometry.Rectangle([b[1], b[0], b[3], b[2]]), 'view', gen=scan['gen'])
    finally:
        scan['busy'] = False

def clear_all(_):
    """Remove drawn boxes and the AI result."""
    if drawn['ctrl']:
        drawn['ctrl'].clear()
    drawn['box'], scan['last'], result['water'] = None, None, None
    show_result()                                       # nothing stored, so this just removes the layer
    say('Boxes and AI results cleared.')

# --- AI controls ---
btn_view = widgets.Button(description='Scan view', icon='refresh')
btn_view.add_class('pill')
btn_view.on_click(water_in_view)

btn_auto = widgets.ToggleButton(description='Auto AI', icon='magic')
btn_auto.add_class('pill')

def toggle_auto(change):
    scan['auto'] = change['new']
    say('Auto AI on: the view is scanned automatically each time you finish moving the map '
        '(zoom in far enough first).' if change['new'] else 'Auto AI off.')
btn_auto.observe(toggle_auto, names='value')

btn_box = widgets.Button(description='AI: drawn box', icon='square-o')
btn_box.add_class('pill')
btn_box.on_click(water_in_box)

btn_clear = widgets.Button(description='Clear', icon='eraser')
btn_clear.add_class('pill')
btn_clear.on_click(clear_all)

# --- Classification filters: redraw the last result without re-scanning ---
filter_specs = [('All types',   'th',           ALL_TYPES, None),
                ('Only lakes',  'circle',       (LAKE,),   'lakes'),
                ('Only rivers', 'road',         (RIVER,),  'rivers'),
                ('Only ponds',  'dot-circle-o', (POND,),   'ponds')]
class_buttons = []

def set_filter(active_btn):
    """Highlight the chosen filter button (None = All types) and redraw."""
    active_btn = active_btn or class_buttons[0]
    for b in class_buttons:
        b.remove_class('active')
    active_btn.add_class('active')
    show_result(active_btn._types)

def make_filter_handler(btn, name):
    def handler(_):
        if result['water'] is None:
            say('Scan the view or a box first, then filter by type.')
            return
        if result['classes'] is None:
            say('This result was scanned too far out to sort by type: zoom in and scan again.')
            return
        set_filter(btn)
        say(f'Showing only {name}.' if name else 'Showing all water types.')
    return handler

for label, icon, types, name in filter_specs:
    b = widgets.Button(description=label, icon=icon)
    b._types = types                                    # which classes this button shows
    b.add_class('pill')
    b.on_click(make_filter_handler(b, name))
    class_buttons.append(b)
class_buttons[0].add_class('active')

# --- Toolbar: one grouped row under the map ---
for b in btns + [btn_view, btn_auto, btn_box, btn_clear] + class_buttons:
    b.layout = widgets.Layout(width='auto')             # size each button to its label (no cut-off text)

def caption(text):
    """Small grey group heading."""
    return widgets.HTML(f'<span style="color:#5f6368;font-size:12px;font-weight:500;'
                        f'letter-spacing:.5px;margin-right:8px">{text.upper()}</span>')

def divider():
    """Thin vertical line between groups."""
    return widgets.HTML('<div style="width:1px;height:28px;background:#dadce0;margin:0 12px"></div>')

toolbar = widgets.HBox(
    [caption('Reference'), *btns, divider(),
     caption('AI'), btn_view, btn_auto, btn_box, btn_clear, divider(),
     caption('Filter'), *class_buttons],
    layout=widgets.Layout(margin='12px 0 0 0', align_items='center', flex_flow='row wrap'))

# --- Assemble: chat | map, toolbar underneath  ---
app = widgets.VBox([css, widgets.HBox([chat_panel, map_area]), toolbar])
app.add_class('app')
app